# Open Food Facts (Big Data) — NOVA Group Prediction (PySpark)




## 0) Colab Setup

In [ ]:
# -------------------
# COLAB SETUP (Run once)
# -------------------
# !apt-get -qq update
# !apt-get -qq install openjdk-11-jdk
# !pip -q install pyspark==3.5.1 mlflow==2.14.3 requests


## 1) Imports

In [ ]:
import os, time, pickle
import numpy as np
import pandas as pd

from huggingface_hub import hf_hub_download

from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array
from pyspark.sql.types import NumericType
from pyspark.sql import SparkSession

from pyspark.ml import Pipeline, Transformer
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, PCA
from pyspark.ml.classification import (
    LogisticRegression, RandomForestClassifier, DecisionTreeClassifier,
    GBTClassifier, OneVsRest
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

from pyspark.mllib.evaluation import MulticlassMetrics


## 2) Spark Session + Config

In [ ]:
# 2) Spark Session + Config (Colab + Spark UI + Event Logs)
import os, shutil
from pyspark.sql import SparkSession

# Where Spark will write event logs (used by History Server / evidence)
EVENT_DIR = "/content/spark-events"
if os.path.exists(EVENT_DIR):
    shutil.rmtree(EVENT_DIR)
os.makedirs(EVENT_DIR, exist_ok=True)

# Where Spark checkpointing will write
CHECKPOINT_DIR = "/content/spark-checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("OpenFoodFacts_NOVA_Classification")
    .master("local[*]")  # enables parallelism across CPU cores
    # memory / stability
    .config("spark.driver.memory", "6g")
    .config("spark.driver.maxResultSize", "512m")
    # performance tuning
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.files.maxPartitionBytes", str(32 * 1024 * 1024))  # 32MB
    # Spark UI
    .config("spark.ui.enabled", "true")
    .config("spark.ui.port", "4040")
    # Event Logging
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", f"file:{EVENT_DIR}")
    .config("spark.eventLog.compress", "false")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")
sc.setCheckpointDir(CHECKPOINT_DIR)

print("Spark version:", spark.version)
print("Spark UI internal:", spark.sparkContext.uiWebUrl)
print("Event log dir:", EVENT_DIR)
print("Checkpoint dir:", CHECKPOINT_DIR)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))

## 3) Load Data (raw_df)
Let's only load **needed columns** first (column pruning).

In [ ]:
from huggingface_hub import hf_hub_download
from pyspark.sql import functions as F

# Load Open Food Facts parquet (big dataset)
path = hf_hub_download(
    repo_id="openfoodfacts/product-database",
    filename="food.parquet",
    repo_type="dataset"
)

needed_cols = [
    "nova_group", "code", "additives_n", "ingredients_n", "nutriments",
    "product_name", "brands", "categories", "countries", "packaging", "labels"
]

_base = spark.read.parquet(path)
raw_df = _base.select([c for c in needed_cols if c in _base.columns])

# Early filter (lazy; reduces later work)
raw_df = raw_df.filter(F.col("nova_group").isNotNull())

print("Loaded columns:", raw_df.columns)
raw_df.printSchema()
raw_df.limit(5).show(truncate=False)

# Open Spark UI after first action
from google.colab import output
import re

ui_url = spark.sparkContext.uiWebUrl
print("Spark UI internal URL:", ui_url)

m = re.search(r":(\d+)", ui_url or "")
port = int(m.group(1)) if m else 4040

print("Opening Spark UI on port:", port)
output.serve_kernel_port_as_iframe(port)


## 4) Ingestion Validation (raw stage)
In this stage, **validate schema**,
**missing target**, **duplicates** are addressed, followed by comprehensive feature validation.

In [ ]:
from pyspark.sql import functions as F
import os
import pandas as pd

print("=== Ingestion validation (Colab-safe) ===")

# Big-data evidence: file size (>=1GB requirement)
size_bytes = os.path.getsize(path)
print("Downloaded parquet size (GB):", round(size_bytes / (1024**3), 3))

print("Columns loaded:", raw_df.columns)

# Safe preview (no full scans)
raw_df.select("nova_group", "code", "additives_n", "ingredients_n").limit(20).show(truncate=False)

# Safe NOVA distribution preview using Python-side counts (avoids Spark shuffle on huge raw_df)
nova_sample = raw_df.select("nova_group").limit(2000).toPandas()
print("Preview NOVA distribution (first 2000 rows):")
print(nova_sample["nova_group"].value_counts(dropna=False).sort_index())


## 5) Working Dataset Creation
Colab local Spark (12GB RAM) is capable of "out of memory" OOM on full parquet scans and writes. As a result, let's document dataset file size as big-data proof and build a **large working sample** `work_df` for model development.


In [ ]:
from pyspark.sql import functions as F
from pyspark import StorageLevel

# Minimal columns required for ML pipeline
raw_min = raw_df.select("nova_group", "code", "additives_n", "ingredients_n", "nutriments")

# Start small and increase if stable (0.01 → 0.05 → 0.10)
WORK_FRAC = 0.02

# avoid counting or persisting while nutrients are still present since this may cause OOM.
work_df = raw_min.sample(withReplacement=False, fraction=WORK_FRAC, seed=42)

# Safe preview (without a complete scan)
work_df.limit(5).show(truncate=False)
print("WORK_FRAC =", WORK_FRAC)
print("Since, nutriments column is very large, so let's skip work_df.count() to avoid OOM.")

# ---- Feature extraction: nutriments is array<struct<name,value,...>> ----
# Safe extractor: returns NULL if nutriment not found
def nutr_value_safe(nutr_name: str):
    arr = F.expr(f"filter(nutriments, x -> x.name = '{nutr_name}')")
    return F.when(
        F.size(arr) > 0,
        F.element_at(arr, 1).value.cast("double")
    ).otherwise(F.lit(None).cast("double"))

# using names i.e. removing _100g that comes under "nutriments"
nutri_features = {
    "energy_100g": "energy",
    "fat_100g": "fat",
    "sugars_100g": "sugars",
    "salt_100g": "salt",
    "proteins_100g": "proteins",
    "carbohydrates_100g": "carbohydrates"
}

df = work_df.withColumn("nova_group", F.col("nova_group").cast("int"))

for out_col, nutr_name in nutri_features.items():
    df = df.withColumn(out_col, nutr_value_safe(nutr_name))

# After extraction, fill missing numeric values
fill_map = {c: 0.0 for c in nutri_features.keys()}
fill_map.update({"additives_n": 0.0, "ingredients_n": 0.0})
df = df.fillna(fill_map)

# Remove invalid negatives
for c in nutri_features.keys():
    df = df.filter(F.col(c) >= 0)

# Drop nutriments (big column) before caching
optimized_df = df.select(
    "nova_group", "code", "additives_n", "ingredients_n",
    *list(nutri_features.keys())
)

# now it is safe to persist + count
optimized_df = optimized_df.persist(StorageLevel.MEMORY_AND_DISK)

optimized_df.limit(5).show(truncate=False)
print("optimized_df columns:", optimized_df.columns)
print("optimized_df rows:", optimized_df.count())

## 6) Feature Stage: Full Validation on optimized_df
Now let's validate **all ML feature columns** because they exist only after feature extraction.

In [ ]:
target_col = "nova_group"

# Except target: Auto-detect numeric features
feature_cols = [
    f.name for f in optimized_df.schema.fields
    if isinstance(f.dataType, NumericType) and f.name != target_col
]

print("Detected feature columns:", feature_cols)

total_rows = optimized_df.count()

# Null % per selected column
null_stats = []
for c in [target_col] + feature_cols:
    null_count = optimized_df.filter(F.col(c).isNull()).count()
    null_stats.append((c, null_count, (null_count / total_rows) * 100))

null_df = spark.createDataFrame(null_stats, ["column", "null_count", "null_percent"])
print("Null counts:")
null_df.show()

# Sanity checks: negatives + extremely large values
sanity_rows = []
for c in feature_cols:
    neg_count = optimized_df.filter(F.col(c) < 0).count()
    big_count = optimized_df.filter(F.col(c) > 10000).count()
    sanity_rows.append((c, neg_count, big_count))

sanity_df = spark.createDataFrame(sanity_rows, ["column", "negative_count", "too_large_count"])
print("Sanity checks:")
sanity_df.show()

# Min/Max summary
summary_exprs = []
for c in feature_cols:
    summary_exprs += [F.min(c).alias(f"{c}_min"), F.max(c).alias(f"{c}_max")]

minmax_row = optimized_df.select(*summary_exprs).collect()[0].asDict()
minmax_long = []
for c in feature_cols:
    min_val = minmax_row.get(f"{c}_min")
    max_val = minmax_row.get(f"{c}_max")
    minmax_long.append((c, float(min_val) if min_val is not None else None,
                           float(max_val) if max_val is not None else None))
minmax_df = spark.createDataFrame(minmax_long, ["column", "min_value", "max_value"])

# Duplicates
dup_count = None
if "code" in optimized_df.columns:
    dup_count = optimized_df.count() - optimized_df.select("code").distinct().count()

print("Total rows:", total_rows)
print("Duplicate count (code):", dup_count)

# Export for Tableau Dashboard 1
null_df.coalesce(1).write.mode("overwrite").option("header", True).csv("tableau_null_summary.csv")
sanity_df.coalesce(1).write.mode("overwrite").option("header", True).csv("tableau_sanity_checks.csv")
minmax_df.coalesce(1).write.mode("overwrite").option("header", True).csv("tableau_minmax_summary.csv")


## 7) Skip full parquet rewrite
It is possible to OOM on Colab by writing the entire Open Food Facts parquet back out so, let's use `optimized_df` as the curated dataset for machine learning.


In [ ]:
# Colab-safe: treating optimized_df as curated dataset
curated_df = optimized_df
print("Curated dataset ready (sampled):")
curated_df.limit(3).show(truncate=False)


## 8) Domain Specific Custom Transformer
Example: simple nutrient score. Replace weights if there is a better domain justification.

In [ ]:
# Import required classes to define a custom Spark ML transformer
from pyspark.ml.param.shared import Param, Params
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable
from pyspark.ml import Transformer
import pyspark.sql.functions as F


# Custom transformer to create a new nutrition-based feature
# This adds a weighted nutrient score using sugar, salt and fat content.
class NutrientScoreTransformer(Transformer, DefaultParamsReadable, DefaultParamsWritable):

    def __init__(self, outputCol="nutrient_score"):
        super().__init__()
        # Define output column parameter
        self.outputCol = Param(self, "outputCol", "Output column name")
        self._setDefault(outputCol=outputCol)
        self._set(outputCol=outputCol)

    # This method runs when the transformer is applied in the pipeline
    # It creates a new column calculated from existing nutritional values
    def _transform(self, dataset):
        out = self.getOrDefault(self.outputCol)
        return dataset.withColumn(
            out,
            (F.col("sugars_100g") * 0.4) +
            (F.col("salt_100g") * 0.3) +
            (F.col("fat_100g") * 0.3)
        )


# Initialise the custom transformer
cust_trans = NutrientScoreTransformer(outputCol="nutrient_score")

## 9) Train/Test split + checkpoint (lineage cutting)

In [ ]:
# stratification in Spark is limited so, let's keep a fixed seed and show class distributions.
train_df, test_df = curated_df.randomSplit([0.8, 0.2], seed=42)

# Cut lineage before expensive model training
train_df_ckpt = train_df.checkpoint(eager=True)
test_df_ckpt = test_df.checkpoint(eager=True)

print("Train rows:", train_df_ckpt.count(), "Test rows:", test_df_ckpt.count())

# Export class distributions per split
train_dist = train_df_ckpt.groupBy("nova_group").count().orderBy("nova_group")
test_dist = test_df_ckpt.groupBy("nova_group").count().orderBy("nova_group")
train_dist.coalesce(1).write.mode("overwrite").option("header", True).csv("class_distribution_train.csv")
test_dist.coalesce(1).write.mode("overwrite").option("header", True).csv("class_distribution_test.csv")


## 10) Common ML Pipeline (index → features → scaler → PCA)
In this stage, let's build a single reusable feature pipeline, then swap different classifiers.

In [ ]:
indexer = StringIndexer(inputCol="nova_group", outputCol="label", handleInvalid="skip")

# Use detected numeric features + engineered score
# Ensure features exist in curated_df:
base_features = [c for c in feature_cols if c != "label"]  # from earlier auto-detect

# Add engineered feature produced by transformer
assembler = VectorAssembler(inputCols=base_features + ["nutrient_score"], outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled", withMean=True, withStd=True)
pca = PCA(k=5, inputCol="features_scaled", outputCol="features")

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")


## 11) Train 4 MLlib models
Models: Logistic Regression, Random Forest, Decision Tree, OvR(GBT).

In [ ]:
# Define models
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=50)
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=200)
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=10)
gbt = GBTClassifier(featuresCol="features", labelCol="label", maxIter=50)  # binary by default; use OvR for multiclass
ovr_gbt = OneVsRest(classifier=gbt, featuresCol="features", labelCol="label")

# Fit pipelines without tuning
trained_pipes = {}

for name, clf in [("LR", lr), ("RF", rf), ("DT", dt), ("OvR_GBT", ovr_gbt)]:
    pipe = Pipeline(stages=[indexer, cust_trans, assembler, scaler, pca, clf])
    print(f"Training {name} ...")
    trained_pipes[name] = pipe.fit(train_df_ckpt)

print("Trained models:", list(trained_pipes.keys()))


## 12) Hyperparameter tuning (CrossValidator) for at least 1 model
Tune LR (and optionally RF) using distributed CV.

In [ ]:
# Logistic Regression model with base settings
lr_tune = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=50
)

# Build full ML pipeline:
# 1) Convert label to indexed format
# 2) Create custom nutrient score
# 3) Assemble features
# 4) Scale features
# 5) Apply PCA (dimensionality reduction)
# 6) Train Logistic Regression
lr_pipe = Pipeline(stages=[indexer, cust_trans, assembler, scaler, pca, lr_tune])

# Define hyperparameter search space
# regParam controls regularisation strength
# elasticNetParam controls L1 vs L2 balance
paramGrid = (ParamGridBuilder()
             .addGrid(lr_tune.regParam, [0.1, 0.01])
             .addGrid(lr_tune.elasticNetParam, [0.0, 0.5])
             .build())

# CrossValidator performs k-fold validation
# numFolds=2 keeps computation manageable
# parallelism=2 allows parallel training of parameter combinations
cv = CrossValidator(
    estimator=lr_pipe,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator.setMetricName("f1"),
    numFolds=2,
    parallelism=2
)

print("Training tuned LR (CV)...")

# Open Spark UI BEFORE CrossValidator
ui_url = spark.sparkContext.uiWebUrl
print("Spark UI internal URL:", ui_url)

m = re.search(r":(\d+)", ui_url or "")
port = int(m.group(1)) if m else 4040

print("Opening Spark UI on port:", port)
output.serve_kernel_port_as_window(port)

# Train model using cross-validation
# The best parameter combination is selected automatically
best_lr_model = cv.fit(train_df_ckpt)

## 13) Serialization

Let's save the best model.



In [ ]:
best_model_path = "best_food_model_spark"
best_lr_model.bestModel.write().overwrite().save(best_model_path)
print("Saved Spark best model to:", best_model_path)

## 14) Evaluate ALL models with 5 metrics
Let's evaluate each model on the same test set and export a single comparison CSV.

In [ ]:
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.sql import functions as F
import pandas as pd

eval_metrics = ["accuracy", "weightedPrecision", "weightedRecall", "f1"]

def macro_f1_from_preds(pred_df, label_col="label", pred_col="prediction"):
    # Get label values that actually appear in this evaluation set
    label_list = [float(r[0]) for r in pred_df.select(label_col).distinct().collect()]

    rdd = pred_df.select(pred_col, label_col).rdd.map(lambda r: (float(r[0]), float(r[1])))
    metrics = MulticlassMetrics(rdd)

    f1s = [metrics.fMeasure(l) for l in label_list]
    return float(sum(f1s) / len(f1s)) if f1s else 0.0

comparison_rows = []

# Add tuned model as well
all_models = dict(trained_pipes)
all_models["LR_TUNED"] = best_lr_model.bestModel

for model_name, model_obj in all_models.items():
    preds = model_obj.transform(test_df_ckpt)

    row = {"Model": model_name}
    for m in eval_metrics:
        row[m] = float(evaluator.setMetricName(m).evaluate(preds))

    # 5th metric
    row["macroF1"] = macro_f1_from_preds(preds)

    comparison_rows.append(row)

comparison_pdf = pd.DataFrame(comparison_rows).sort_values("f1", ascending=False)
comparison_pdf.to_csv("all_models_metrics_5.csv", index=False)

print("Saved: all_models_metrics_5.csv")
comparison_pdf

## 15) NOVA-4 recall per model
Confirm label mapping from StringIndexer. NOVA=4 may not equal label 3 depending on label order.

In [ ]:
# Extract the trained StringIndexer from the best tuned LR pipeline
# This shows how original nova_group values were mapped to numeric labels
indexer_model = best_lr_model.bestModel.stages[0]
labels_order = indexer_model.labels  # original nova_group values as strings
print("StringIndexer label order:", labels_order)

# Identify which numeric label corresponds to NOVA group 4
# This is needed because Spark internally converts class labels to indices
nova4_label_value = None
if "4" in labels_order:
    nova4_label_value = float(labels_order.index("4"))
print("Label index for NOVA=4:", nova4_label_value)


# Function to calculate recall for a specific class
# Uses Spark's MulticlassMetrics on prediction output
def per_class_recall(pred_df, target_label, label_col="label", pred_col="prediction"):
    rdd = pred_df.select(pred_col, label_col).rdd.map(lambda r: (float(r[0]), float(r[1])))
    metrics = MulticlassMetrics(rdd)
    return float(metrics.recall(float(target_label)))


# Compute recall for NOVA group 4 across all trained models
# This helps evaluate how well each model detects ultra-processed foods
rows = []
if nova4_label_value is not None:
    for model_name, model_obj in all_models.items():
        preds = model_obj.transform(test_df_ckpt)
        rows.append((model_name, per_class_recall(preds, nova4_label_value)))

# Save results for reporting and Tableau dashboard
nova4_df = pd.DataFrame(rows, columns=["Model", "Recall_NOVA4"])
nova4_df.to_csv("nova4_recall_by_model.csv", index=False)
print("Saved: nova4_recall_by_model.csv")

nova4_df

## 16) Bootstrap Confidence Interval for best model (percentile CI)

In [ ]:
# Identify the best performing model based on the comparison table
best_model_name = comparison_pdf.iloc[0]["Model"]
best_model = all_models[best_model_name]

# Generate predictions from the best model on the test dataset
# Only keep label and prediction columns for evaluation
best_preds = best_model.transform(test_df_ckpt).select("label", "prediction").cache()

# Bootstrap settings
# B defines the number of resampling iterations
B = 50
scores = []

# Perform bootstrap resampling on prediction results
# Each iteration samples with replacement and calculates F1 score
for i in range(B):
    sample = best_preds.sample(withReplacement=True, fraction=1.0, seed=100+i)
    scores.append(float(evaluator.setMetricName("f1").evaluate(sample)))

# Convert results to numpy array for statistical calculations
scores = np.array(scores)

# Compute 95% confidence interval using percentiles
ci_low = float(np.percentile(scores, 2.5))
ci_high = float(np.percentile(scores, 97.5))

# Store bootstrap summary statistics
ci_df = pd.DataFrame([{
    "Model": best_model_name,
    "Metric": "f1",
    "Mean": float(scores.mean()),
    "Std": float(scores.std()),
    "CI_2.5": ci_low,
    "CI_97.5": ci_high,
    "B": B
}])

# Save results for reporting
ci_df.to_csv("bootstrap_ci_bestmodel.csv", index=False)
print("Saved: bootstrap_ci_bestmodel.csv")

ci_df

## 17) scikit-learn baseline (single node) + pickle
Due to the single-node nature of sklearn, sample a limited fraction. Compare runtime and metrics.

In [ ]:
# Single-node baseline comparison using sklearn
# This section trains Logistic Regression and Random Forest on a small pandas sample to compare distributed Spark ML
# performance with a traditional single-machine approach.

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression as SkLR
from sklearn.ensemble import RandomForestClassifier as SkRF


# Convert a small Spark sample to pandas
# Keeping the fraction small avoids memory issues in Colab
sample_pd = (curated_df
             .sample(withReplacement=False, fraction=0.01, seed=42)
             .select("nova_group", *feature_cols)
             .toPandas()
             .dropna())

# Separate features (X) and target labels (y)
y = sample_pd["nova_group"].astype(int).values
X = sample_pd[feature_cols].astype(float).values

# Split into training and testing sets
# Stratified split preserves class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rows = []

# -------------------------------------------------------
# sklearn Logistic Regression (with feature scaling)
# -------------------------------------------------------

# Pipeline ensures scaling is applied before training
lr_pipe = SkPipeline([
    ("scaler", StandardScaler()),
    ("clf", SkLR(max_iter=500, n_jobs=-1, multi_class="auto"))
])

# Measure training time
t0 = time.time()
lr_pipe.fit(X_train, y_train)
t_train = time.time() - t0

# Measure inference time
t0 = time.time()
pred = lr_pipe.predict(X_test)
t_infer = time.time() - t0

# Save trained model
with open("sklearn_lr_baseline.pkl", "wb") as f:
    pickle.dump(lr_pipe, f)

# Store evaluation results
rows.append({
    "Model": "sklearn_LR",
    "accuracy": accuracy_score(y_test, pred),
    "weightedPrecision": precision_score(y_test, pred, average="weighted", zero_division=0),
    "weightedRecall": recall_score(y_test, pred, average="weighted", zero_division=0),
    "weightedF1": f1_score(y_test, pred, average="weighted", zero_division=0),
    "train_time_sec": t_train,
    "inference_time_sec": t_infer,
    "n_train": len(y_train),
    "n_test": len(y_test)
})

# -------------------------------------------------------
# sklearn Random Forest (scaling not required)
# -------------------------------------------------------

sk_rf = SkRF(n_estimators=200, n_jobs=-1, random_state=42)

# Measure training time
t0 = time.time()
sk_rf.fit(X_train, y_train)
t_train = time.time() - t0

# Measure inference time
t0 = time.time()
pred2 = sk_rf.predict(X_test)
t_infer = time.time() - t0

# Save trained model
with open("sklearn_rf_baseline.pkl", "wb") as f:
    pickle.dump(sk_rf, f)

# Store evaluation results
rows.append({
    "Model": "sklearn_RF",
    "accuracy": accuracy_score(y_test, pred2),
    "weightedPrecision": precision_score(y_test, pred2, average="weighted", zero_division=0),
    "weightedRecall": recall_score(y_test, pred2, average="weighted", zero_division=0),
    "weightedF1": f1_score(y_test, pred2, average="weighted", zero_division=0),
    "train_time_sec": t_train,
    "inference_time_sec": t_infer,
    "n_train": len(y_train),
    "n_test": len(y_test)
})

# Convert results to DataFrame and export for reporting / Tableau
baseline_df = pd.DataFrame(rows)
baseline_df.to_csv("sklearn_baseline_metrics.csv", index=False)

print("Saved: sklearn_baseline_metrics.csv")
baseline_df

## 18) Scalability analysis
### Strong scaling proxy
Fixed data size, vary shuffle partitions.

### Weak scaling proxy
Increase data fraction; keep configs constant.

In [ ]:
# Function to measure how long a model takes to transform a dataset
# It applies the model and forces execution using count()
def timed_transform_count(model, df):
    start = time.time()
    _ = model.transform(df).count()  # Trigger Spark action
    return time.time() - start


# -------------------------
# Strong Scaling Proxy Test
# -------------------------
# Keep dataset size fixed and vary shuffle partitions
# This simulates how task parallelism affects runtime

strong_rows = []

# Use a small fixed sample to keep runtime reasonable
fixed_df = curated_df.sample(False, 0.05, seed=42).cache()
fixed_df.count()  # Materialise cache

# Test different shuffle partition settings
for sp in [10, 50, 200]:
    spark.conf.set("spark.sql.shuffle.partitions", str(sp))

    # Measure execution time of best model
    dur = timed_transform_count(best_model, fixed_df)

    strong_rows.append({
        "shuffle_partitions": sp,
        "rows": fixed_df.count(),
        "runtime_sec": dur
    })

# Clear cache after experiment
fixed_df.unpersist()

# Convert results to pandas for analysis
strong_pdf = pd.DataFrame(strong_rows)

# Simple cost proxy (runtime × assumed core factor)
strong_pdf["cost_proxy"] = strong_pdf["runtime_sec"] * 2

# Save results for reporting
strong_pdf.to_csv("strong_scaling_proxy.csv", index=False)

print("Saved: strong_scaling_proxy.csv")
strong_pdf

In [ ]:
# -------------------------
# Weak Scaling Proxy Test
# -------------------------
# Increase dataset size while keeping configuration fixed
# This observes how runtime grows as data volume increases

weak_rows = []

for frac in [0.01, 0.03, 0.05, 0.10]:

    # Take a fraction of the dataset
    df_frac = curated_df.sample(False, frac, seed=42).cache()

    # Count rows to record actual dataset size
    rows = df_frac.count()

    # Measure execution time of best model on this subset
    dur = timed_transform_count(best_model, df_frac)

    weak_rows.append({
        "fraction": frac,
        "rows": rows,
        "runtime_sec": dur
    })

    # Clear cache after each iteration
    df_frac.unpersist()

# Convert results to pandas
weak_pdf = pd.DataFrame(weak_rows)

# Simple cost proxy for discussion (runtime × assumed core factor)
weak_pdf["cost_proxy"] = weak_pdf["runtime_sec"] * 2

# Save results for scalability analysis in report
weak_pdf.to_csv("weak_scaling_proxy.csv", index=False)

print("Saved: weak_scaling_proxy.csv")
weak_pdf

In [ ]:
best_model_path = "best_food_model_spark"
best_lr_model.bestModel.write().overwrite().save(best_model_path)
print("Saved Spark best model to:", best_model_path)


## 20) Visualization Exports for Tableau
This section generates **visualization-ready CSVs** for:
- Confusion matrix (best model)
- Classification report (per-class precision/recall/F1/support)
- Feature importance (RandomForest without PCA)


### 20.1 Pick the best model for visualizations
If accessible, use the top model from `all_models_metrics_5.csv`; if not, fall back to `LR_TUNED`.

In [ ]:
# Choose best model based on previously computed comparison_pdf if it exists
try:
    best_model_name = comparison_pdf.iloc[0]["Model"]
except Exception:
    best_model_name = "LR_TUNED"

all_models = dict(trained_pipes)
all_models["LR_TUNED"] = best_lr_model.bestModel

best_model = all_models.get(best_model_name, best_lr_model.bestModel)
print("Best model used for visualization exports:", best_model_name)

best_preds = best_model.transform(test_df_ckpt).cache()
best_preds.select("label","prediction").limit(5).show()


### 20.1a Before vs After Scaling (Normalization)
Let's export a short sample displaying selected numeric features before to scaling and their corresponding outputs from StandardScaler.

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.functions import vector_to_array

# Pick a few numeric features to visualise in Tableau
viz_raw_cols = [c for c in ["sugars_100g","fat_100g","salt_100g","proteins_100g","energy-kcal_100g"] if c in curated_df.columns]

if len(viz_raw_cols) < 2:
    raise ValueError(f"Need at least 2 numeric columns for scaling visuals. Found: {viz_raw_cols}")

# Build a small scaling-only pipeline for visualization (keeps mapping 1:1 with viz_raw_cols)
viz_assembler = VectorAssembler(inputCols=viz_raw_cols, outputCol="viz_features_raw")
viz_scaler = StandardScaler(inputCol="viz_features_raw", outputCol="viz_features_scaled", withMean=True, withStd=True)

# Use train split and take a small random sample for Tableau performance
N_SAMPLE = 5000
viz_df = (
    train_df_ckpt
    .select(*viz_raw_cols)
    .dropna()
    .orderBy(F.rand(seed=42))
    .limit(N_SAMPLE)
)

viz_df = viz_assembler.transform(viz_df)
viz_df = viz_scaler.fit(viz_df).transform(viz_df)

# Split scaled vector into columns: scaled_<colname>
viz_df = viz_df.withColumn("viz_scaled_arr", vector_to_array(F.col("viz_features_scaled")))
for i, c in enumerate(viz_raw_cols):
    viz_df = viz_df.withColumn(f"scaled_{c}", F.col("viz_scaled_arr")[i])

viz_out = viz_df.select(*viz_raw_cols, *[f"scaled_{c}" for c in viz_raw_cols])

out_before_after_scaling = "tableau_before_after_scaling.csv"
viz_out.coalesce(1).write.mode("overwrite").option("header", True).csv(out_before_after_scaling)

print("Saved:", out_before_after_scaling, "Columns:", viz_out.columns)


### 20.1b Baseline (Before Tuning) metrics export
Trains a baseline Logistic Regression model (without CrossValidation) with the identical feature pipeline and exports metrics for comparison with the customized model in Tableau.

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline as SparkPipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 0) Fix assembler input columns
existing_cols = set(train_df_ckpt.columns)

orig_inputs = assembler.getInputCols()
kept_inputs = [c for c in orig_inputs if c in existing_cols]
removed_inputs = [c for c in orig_inputs if c not in existing_cols]

print("Removed missing assembler cols:", removed_inputs)
print("Kept assembler cols:", kept_inputs)

assembler = assembler.setInputCols(kept_inputs)

# 1) Use correct features column
# Get PCA output column safely:
pca_out = pca.getOutputCol()  # this is the correct one from your existing pca stage
print("LR will use featuresCol =", pca_out)

# Baseline LR (no CV / no tuning)
lr_base = LogisticRegression(featuresCol=pca_out, labelCol="label", maxIter=50)

# Reuse the same stages
lr_base_pipe = SparkPipeline(stages=[indexer, assembler, scaler, pca, lr_base])

# Fit + predict
lr_base_model = lr_base_pipe.fit(train_df_ckpt)
lr_base_preds = lr_base_model.transform(test_df_ckpt)  # keep full DF for evaluators

# 2) Evaluate
evals = {
    "accuracy": MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy"),
    "weightedPrecision": MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision"),
    "weightedRecall": MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall"),
    "f1": MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1"),
}

lr_base_metrics = {"Model": "LR_BASE_UNTUNED"}
for k, ev in evals.items():
    lr_base_metrics[k] = float(ev.evaluate(lr_base_preds))

metrics_df = spark.createDataFrame([lr_base_metrics])

# 3) Save to CSV
out_lr_base_metrics = "lr_base_metrics.csv"
(
    metrics_df
    .coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(out_lr_base_metrics)
)

print("Saved:", out_lr_base_metrics)
metrics_df.show(truncate=False)

### 20.2 Confusion Matrix (Multiclass)
Exports:
- `confusion_matrix_long.csv` (label, prediction, count)
- `confusion_matrix_wide.csv` (pivoted matrix for Tableau heatmap)


In [ ]:
# Confusion matrix long format
cm_long = (best_preds
           .groupBy("label", "prediction")
           .count()
           .orderBy("label", "prediction"))

cm_long.show(20, truncate=False)
cm_long.coalesce(1).write.mode("overwrite").option("header", True).csv("confusion_matrix_long.csv")

# Confusion matrix wide (pivot)
cm_wide = (cm_long
           .groupBy("label")
           .pivot("prediction")
           .sum("count")
           .na.fill(0)
           .orderBy("label"))

# Save wide as single CSV via pandas (pivot columns vary)
cm_wide_pd = cm_wide.toPandas()
cm_wide_pd.to_csv("confusion_matrix_wide.csv", index=False)

print("Saved: confusion_matrix_long.csv and confusion_matrix_wide.csv")
cm_wide_pd


### 20.3 Classification Report (per-class metrics)
Exports `classification_report_by_class.csv` with precision/recall/F1/support for each class.

In [ ]:
# Build MulticlassMetrics
rdd = best_preds.select("prediction", "label").rdd.map(lambda r: (float(r[0]), float(r[1])))
m = MulticlassMetrics(rdd)

# Determine class labels present
labels_present = sorted([float(x[0]) for x in best_preds.select("label").distinct().collect()])

rows = []
for l in labels_present:
    rows.append({
        "class_label": l,
        "precision": float(m.precision(l)),
        "recall": float(m.recall(l)),
        "f1": float(m.fMeasure(l)),
        "support": int(best_preds.filter(F.col("label") == l).count())
    })

report_pd = pd.DataFrame(rows)
report_pd.to_csv("classification_report_by_class.csv", index=False)
print("Saved: classification_report_by_class.csv")
report_pd


### 20.4 Precision–Recall Curve for NOVA 4 (One-vs-Rest)
Prints the average precision and exports the threshold-free curve points, `pr_curve_nova4.csv`.

There must be a probability column for this. If probability is not produced by the selected best model, revert to `LR_TUNED`.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

# Check whether probability column exists in predictions
# If not available (e.g. tree-based model), switch to tuned Logistic Regression
if "probability" not in best_preds.columns:
    print("No probability column in best model predictions; falling back to LR_TUNED.")
    best_model_name = "LR_TUNED"
    best_model = best_lr_model.bestModel
    best_preds = best_model.transform(test_df_ckpt).cache()


# Identify numeric index corresponding to NOVA group 4
# This is required because Spark converts labels to indexed format
nova4_label_index = None
try:
    indexer_model = best_lr_model.bestModel.stages[0]  # StringIndexerModel
    labels_order = indexer_model.labels
    print("StringIndexer labels order:", labels_order)
    if "4" in labels_order:
        nova4_label_index = float(labels_order.index("4"))
except Exception as e:
    print("Could not infer NOVA4 label index from StringIndexerModel:", e)

# Fallback value in case mapping cannot be retrieved
if nova4_label_index is None:
    nova4_label_index = 3.0
    print("WARNING: Using fallback nova4_label_index=3.0. Verify StringIndexer label mapping.")


# Extract probability values for NOVA4 class
# Convert Spark vector column into array for indexing
preds_scored = (best_preds
                .withColumn("prob_arr", vector_to_array("probability"))
                .withColumn("y_true", (F.col("label") == F.lit(nova4_label_index)).cast("int"))
                .withColumn("y_score", F.col("prob_arr")[int(nova4_label_index)]))


# Limit rows before converting to pandas (avoids memory issues)
MAX_ROWS = 200000
pdf = preds_scored.select("y_true", "y_score").limit(MAX_ROWS).toPandas()

# Prepare arrays for sklearn PR calculation
y_true = pdf["y_true"].values
y_score = pdf["y_score"].values


# Compute Precision-Recall curve and Average Precision
prec, rec, thr = precision_recall_curve(y_true, y_score)
ap = average_precision_score(y_true, y_score)

# Save PR curve data for reporting / Tableau
pr_pd = pd.DataFrame({
    "precision": prec,
    "recall": rec
})
pr_pd.to_csv("pr_curve_nova4.csv", index=False)

print("Saved: pr_curve_nova4.csv")
print("Average Precision (NOVA4 OVR):", float(ap))

pr_pd.head()

### 20.5 Feature Importance
Training an RF pipeline **without PCA** to keep importance interpretable and export `feature_importance_rf.csv`.

In [ ]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline as SparkPipeline
from pyspark.ml.feature import StandardScaler as SparkStandardScaler
from sklearn.preprocessing import StandardScaler as SkStandardScaler
from sklearn.pipeline import Pipeline as SkPipeline

# If base_features is not already defined, create it
# Exclude target and identifier columns
try:
    _base_features = base_features
except NameError:
    cols = [c for c in optimized_df.columns if c not in ["nova_group", "code"]]
    _base_features = cols  # includes additives_n, ingredients_n and numeric nutriments


# Random Forest model used specifically to extract feature importance
rf_imp = RandomForestClassifier(featuresCol="features_raw", labelCol="label", numTrees=200)

# Index target variable into numeric label format
indexer_imp = StringIndexer(inputCol="nova_group", outputCol="label", handleInvalid="skip")

# Combine original features + custom nutrient score into one vector
assembler_imp = VectorAssembler(
    inputCols=_base_features + ["nutrient_score"],
    outputCol="features_raw"
)

# Standardise features before training
scaler_imp = SparkStandardScaler(
    inputCol="features_raw",
    outputCol="features_scaled",
    withMean=True,
    withStd=True
)


# Final Random Forest model using scaled features
rf_imp2 = RandomForestClassifier(
    featuresCol="features_scaled",
    labelCol="label",
    numTrees=200
)

# Build pipeline without PCA (important: keeps original feature meaning)
rf_pipe = SparkPipeline(stages=[indexer_imp, cust_trans, assembler_imp, scaler_imp, rf_imp2])

print("Training RF for feature importance (no PCA)...")

# Train model
rf_model = rf_pipe.fit(train_df_ckpt)

# Extract trained RF stage from pipeline
rf_stage = rf_model.stages[-1]

# Retrieve feature importance scores
importances = rf_stage.featureImportances

# Map importance values back to feature names
feat_names = assembler_imp.getInputCols()
imp_rows = [
    {"feature": f, "importance": float(importances[i])}
    for i, f in enumerate(feat_names)
]

# Convert to pandas and sort descending
imp_pd = pd.DataFrame(imp_rows).sort_values("importance", ascending=False)

# Save results for reporting and dashboard
imp_pd.to_csv("feature_importance_rf.csv", index=False)

print("Saved: feature_importance_rf.csv")

imp_pd.head(20)

### 20.6 PCA Explained Variance
Exports `pca_explained_variance.csv` from the tuned LR pipeline.

In [ ]:
from pyspark.ml.feature import PCAModel

# Try to find a fitted PCAModel inside the tuned LR pipeline
pca_model = None

def find_pca_model(pipeline_model):
    for st in pipeline_model.stages:
        if isinstance(st, PCAModel):
            return st
    return None

try:
    pca_model = find_pca_model(best_lr_model.bestModel)
except Exception:
    pca_model = None

if pca_model is None:
    # Try from one of the trained pipelines
    try:
        any_model = next(iter(trained_pipes.values()))
        pca_model = find_pca_model(any_model)
    except Exception:
        pca_model = None

if pca_model is None:
    print("No PCAModel found in pipeline stages. If you removed PCA, you can skip this section.")
else:
    ev = np.array(pca_model.explainedVariance.toArray())
    cum = np.cumsum(ev)

    pca_pd = pd.DataFrame({
        "component": np.arange(1, len(ev)+1),
        "explained_variance": ev,
        "cumulative_variance": cum
    })
    pca_pd.to_csv("pca_explained_variance.csv", index=False)
    print("Saved: pca_explained_variance.csv")
    pca_pd


## 21) Visualization Images (PNG)

This section **creates PNG images** (and displays them) using the CSV/Pandas outputs from Section 20.

Files created:
- `confusion_matrix.png`
- `classification_report_heatmap.png`
- `feature_importance_rf.png`
- `model_comparison_f1.png`


### 21.1 Confusion Matrix Heatmap (PNG)

In [ ]:
import matplotlib.pyplot as plt

# Requires: cm_wide_pd (created in Section 20.2)
try:
    cm_plot = cm_wide_pd.copy()
except NameError:
    cm_plot = pd.read_csv("confusion_matrix_wide.csv")

cm_plot = cm_plot.set_index("label")

plt.figure(figsize=(7,6))

# Blue-white colormap
plt.imshow(cm_plot.values, cmap="Blues", aspect="auto")

plt.title("Confusion Matrix (Best Model)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.xticks(
    ticks=np.arange(cm_plot.shape[1]),
    labels=cm_plot.columns,
    rotation=0
)

plt.yticks(
    ticks=np.arange(cm_plot.shape[0]),
    labels=cm_plot.index
)

# Add values inside cells
for i in range(cm_plot.shape[0]):
    for j in range(cm_plot.shape[1]):
        plt.text(
            j, i,
            int(cm_plot.values[i, j]),
            ha="center",
            va="center",
            color="black"
        )

# Clean style (no grid)
plt.grid(False)

plt.colorbar()
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()

print("Saved: confusion_matrix.png")

### 21.2 Classification Report Heatmap (PNG)

In [ ]:
# Requires: report_pd (created in Section 20.3)
try:
    rep = report_pd.copy()
except NameError:
    rep = pd.read_csv("classification_report_by_class.csv")

heat = rep[["class_label","precision","recall","f1"]].set_index("class_label")

plt.figure(figsize=(7,4))
plt.imshow(heat.values, aspect="auto")
plt.title("Classification Report Heatmap (Best Model)")
plt.xlabel("Metric")
plt.ylabel("Class label")
plt.xticks(ticks=np.arange(heat.shape[1]), labels=heat.columns, rotation=0)
plt.yticks(ticks=np.arange(heat.shape[0]), labels=heat.index)

for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        plt.text(j, i, f"{heat.values[i, j]:.2f}", ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.savefig("classification_report_heatmap.png", dpi=300)
plt.show()
print("Saved: classification_report_heatmap.png")


### 21.4 Feature Importance (PNG)

In [ ]:
# Requires: imp_pd (created in Section 20.5)
try:
    imp = imp_pd.copy()
except NameError:
    imp = pd.read_csv("feature_importance_rf.csv")

top_n = 15
imp_top = imp.sort_values("importance", ascending=False).head(top_n).iloc[::-1]

plt.figure(figsize=(8,6))
plt.barh(imp_top["feature"], imp_top["importance"])
plt.title(f"Top {top_n} Feature Importances (RF, no PCA)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("feature_importance_rf.png", dpi=300)
plt.show()
print("Saved: feature_importance_rf.png")


### 21.5 Model Comparison (Weighted F1) (PNG)

In [ ]:
# Requires: comparison_pdf (created earlier)
try:
    cmp = comparison_pdf.copy()
except NameError:
    cmp = pd.read_csv("all_models_metrics_5.csv")

cmp_sorted = cmp.sort_values("f1", ascending=True)

plt.figure(figsize=(8,5))
plt.barh(cmp_sorted["Model"], cmp_sorted["f1"])
plt.title("Model Comparison (Weighted F1)")
plt.xlabel("Weighted F1")
plt.ylabel("Model")
plt.tight_layout()
plt.savefig("model_comparison_f1.png", dpi=300)
plt.show()
print("Saved: model_comparison_f1.png")


**Outputs created for Tableau:**
- `tableau_null_summary.csv`, `tableau_sanity_checks.csv`, `tableau_minmax_summary.csv`
- `class_distribution.csv`
- `all_models_metrics_5.csv`, `nova4_recall_by_model.csv`, `bootstrap_ci_bestmodel.csv`
- `sklearn_baseline_metrics.csv`
- `tableau_before_after_scaling.csv`
- `lr_base_metrics.csv`
- `strong_scaling_proxy.csv`, `weak_scaling_proxy.csv`
